# Hermes Agent 实战篇：技术博客自动发布系统


## 把前四篇串成一个能交接、能审核、能维护的案例

这篇围绕本地《05-实战篇.md》的综合案例展开：用 Hermes 维护一个技术博客，从发现选题、调研和写稿，到审核、提交文章与站点维护。

前四篇分别介绍了安装与模型、工具与记忆、技能与定时任务、消息入口与多角色协作。这里需要把它们接起来，回答一个具体问题：**一篇文章怎样从“想写”变成“已经上线”，中间每一步由谁完成，结果存在哪里？**

| 课程场景 | 本篇采用的约定 |
| --- | --- |
| 两人技术团队，每周发布 2～3 篇文章 | 定时收集素材，由负责人选择值得写的主题 |
| 主题围绕 AI 与前端开发 | 调研时保留版本、原始来源和核查日期 |
| Hugo + GitHub Pages | Markdown 文章进入仓库，合并后由项目部署流程发布 |
| Telegram 跟进和审批 | 消息入口接收需求，审批记录绑定具体稿件版本 |
| 多 Profile 协作 | researcher、writer、reviewer、publisher 分工，orchestrator 协调 |

排版延续前几篇及“李勃老师零到全栈 7.2”的概念说明、分步示例、表格与复习题。课件以 v0.16.0 为基准；关键接口对照了 2026-10-09 读取的官方文档和公开 main 源码，差异会在对应位置说明。

所有单元格均为 Markdown，示例只供阅读。本篇没有创建 Profile、启动服务、安装插件、设置 Cron、发送消息或操作博客仓库。


## 整体流程：生产文章、传递材料、确认上线

~~~text
Cron 收集新闻 → 筛选选题 → 生成简报 → 人工选择主题
                                         ↓
Telegram / API → Gateway → blog Board → 拆解与调度
                                         ↓
                         技术调研 A + 社区调研 B
                                         ↓
                                  Writer 生成草稿
                                         ↓
                         Reviewer 核查 + 人工批准稿件版本
                                         ↓
                         Publisher 提交文章分支与 PR
                                         ↓
                         合并 → GitHub Actions 部署 → 线上核验

Skills / Curator：维护工作方法    Memory：保存各角色稳定偏好
Hooks：记录运行指标             Dashboard：查看任务与配置
No-Agent Cron：检查站点         Kanban：保存任务、依赖和交接
~~~

这条链有三类不同的状态：Kanban 保存工作进度，文件或附件保存交付内容，Git 与部署系统保存发布版本。聊天里一句“完成了”，不能替代其中任何一项。

本篇统一采用课件前面提出的 **PR → 合并 main → 自动部署** 路线。原文发布示例直接 push main，与前面的要求不一致；这里把提交、合并和上线分开说明。


## 25 项能力分别用在什么地方？

功能清单是检查案例覆盖面的索引，不是要求每篇文章都使用所有工具。

| 能力 | 在博客流程中的用途 | 配合条件 |
| --- | --- | --- |
| Profile | 保存调研、写作、审核、发布各角色配置 | 角色存在，描述清楚 |
| Provider | 为不同角色选择模型与服务 | 配置凭证与实际可用模型 |
| Skills | 加载选题、写作与发布流程 | 技能已安装，依赖已具备 |
| skill_manage | 把验证过的写作方法保存为技能 | 内容带 frontmatter，先审阅再复用 |
| Curator | 检查技能使用状态，维护或归档 | 策略已配置，维护范围明确 |
| Memory | 保存博客格式、语言和发布偏好 | 按 Profile 保存，不存访问令牌 |
| session_search | 查当前角色过去的选题和写作讨论 | 可搜索的会话属于当前作用域 |
| Cron | 定时采集、筛选和生成简报 | 调度入口在线，时区正确 |
| context_from | 把上游已完成输出注入下游任务 | 校验批次和资料日期 |
| Delegation | 在一次调研中并行核对独立问题 | 当前角色具备委派工具与容量 |
| Kanban | 管理任务、依赖、评论与交接 | 明确 Board、assignee 和产物路径 |
| Kanban Swarm | 创建并行研究、核验、综合的图 | 不等同于完整发布流水线 |
| Orchestrator | 管理任务拆解与最终验收 | 区分根任务角色与内置拆解器 |
| Gateway | 接收 Telegram 指令和返回通知 | 连接、路由、准入和订阅分别确认 |
| MCP | 通过 GitHub 工具提交文件或创建 PR | Server 已配置，工具 Schema 已发现 |
| Toolsets | 给角色提供搜索、文件、终端等工具 | 禁用工具列表不等于完整允许清单 |
| Hooks | 记录搜索调用和请求使用量 | 选对事件和字段 |
| Plugins | 扩展文章元数据检查工具 | 在 reviewer 上启用并加载 |
| Dashboard | 查看任务图、运行记录与 Profile 配置 | 选择正确的管理对象 |
| TUI | 交互式操作与查看本会话过程 | 新终端不是其他 Worker 的镜像 |
| API Server | 接收 CMS 的任务请求 | 配置 API 入口与 Bearer 认证 |
| 文件上下文引用 | 给 Writer 提供风格样本 | 入口支持，文件路径能解析 |
| No-Agent Cron | 运行固定站点检查脚本 | 脚本可读取配置，不需要 LLM |
| SOUL.md | 描述各角色的工作风格与职责 | 不替代权限与程序校验 |
| 上下文压缩 | 缩减长会话历史 | 关键交付仍需写到持久记录 |

适合固定脚本判断的事情交给脚本；需要比较、综合和写作的事情交给模型；需要确认发布内容的事情留给审核流程。


## 先约定产物：每个阶段必须交出什么

| 阶段 | 交付内容 | 验收方式 |
| --- | --- | --- |
| 选题 | 主题、受众、价值、与旧文的区别 | 人工选择是否进入写作 |
| 技术调研 | 版本、事实、源码或文档链接 | 关键结论能回到原始来源 |
| 社区调研 | 真实案例、反馈与限制 | 区分反馈、推测和官方说明 |
| 写作 | Markdown 草稿、frontmatter、引用清单 | 内容连贯，示例可静态检查 |
| 审核 | 缺陷、处理结果、稿件路径与 SHA-256 | 审核对应实际文件版本 |
| PR 提交 | 分支、提交、PR URL、构建记录 | PR 中只包含本次文章变更 |
| 上线 | 部署运行、对应提交、线上 URL | 页面可访问且正文属于目标版本 |

“调研完成”“审核通过”“Git 已 push”“部署成功”是四种结果。Kanban 的完成摘要应说明本张卡实际达到了哪一种，避免下游把半成品当成最终交付。

### 完成标准要跟任务标题一致

如果 publisher 卡片叫“提交文章 PR”，创建 PR 后可以完成；如果叫“发布文章并确认线上页面”，只创建 PR 还未达到完成标准。两种任务都合理，关键是不要混用。


## 环境前提：模型、站点与部署流程需要分别准备

本案例假定 Hermes 已按第 01 篇配置完成，博客仓库本身也能在人工操作下构建和部署。

| 前提 | 需要确认的内容 |
| --- | --- |
| 模型服务 | 各 Profile 的 Provider、模型、鉴权和费用预期 |
| 搜索与浏览器 | 搜索后端、网页提取工具及浏览器依赖 |
| Hugo 博客 | 主题、配置、文章路径和构建版本 |
| GitHub 仓库 | 仓库访问、分支规则、PR 与部署工作流 |
| 消息平台 | Bot 配置、允许用户、通知目标 |
| 本地目录 | 工作目录可读写，角色之间有明确文件交接方式 |

后面的 Bash 路径以 Linux / WSL 为例，使用 /srv/blog-work 和 /srv/tech-blog。它们是示例目录，需要替换为实际存在的绝对路径。Windows 原生应使用实际数据目录和 Windows 路径，不能直接照搬 ~/.hermes 或 /srv。

先证明站点能人工发布，再把发布步骤交给 Agent，排查时才容易区分博客工程问题与 Agent 工作流问题。


## 创建五个 Profile：四个执行角色与一个协调角色

~~~bash
hermes profile create researcher --clone --description "核对文档、源码与公开资料，交付带版本和来源的研究摘要。"
hermes profile create writer --clone --description "依据研究材料编写中文技术文章，维护 Markdown 草稿与引用。"
hermes profile create reviewer --clone --description "核查技术准确性、格式、构建与稿件版本，给出具体修改意见。"
hermes profile create publisher --clone --description "依据批准稿件提交博客文章 PR，记录提交、PR 与部署状态。"
hermes profile create orchestrator --clone --description "管理博客任务图、依赖、产物与人工审批，核查最终完成情况。"
hermes profile list
~~~

克隆从当前所选 Profile 复制配置等内容。当前普通 --clone 默认不复制消息 Bot 凭证与准入名单，避免多个进程持同一个 Bot Token 产生冲突；它也不是把旧 Profile 的全部历史与定时任务复制过去。

本篇统一使用 hermes -p NAME 指定身份。课件中的 researcher model 等短命令依赖 Profile wrapper 已创建且能从 PATH 找到，换机器时容易失效。

Profile 分开持久状态，操作系统权限仍由运行账号决定。描述里写“只调研”，并不会自动限制它访问文件系统。


## 模型与工具：按角色配置，别把名称当成能力保证

~~~bash
hermes -p researcher model
hermes -p writer model
hermes -p reviewer model
hermes -p publisher tools list
hermes -p orchestrator tools list
~~~

课件用 DeepSeek 举例说明调研成本，用 Claude 举例说明写作质量。模型选择应按实际可用性、任务效果和预算决定；不同任务需要验证，而不是仅凭模型名称判断。

| 角色 | 常见所需工具 | 尽量避免混入的职责 |
| --- | --- | --- |
| researcher | 搜索、提取、读取资料、按需委派 | 写稿件主文件或提交仓库 |
| writer | 读取材料、文件编辑、按需搜索核查 | 合并 PR 与发布 |
| reviewer | 读取稿件、检查插件、构建核验 | 无记录地重写全文 |
| publisher | 读取批准稿、Git / GitHub 工具 | 自动改变结论或跳过审批 |
| orchestrator | Kanban 管理与必要查询 | 承担所有正文写作 |

例如课件禁用 publisher 的搜索与浏览器：

~~~bash
hermes -p publisher tools disable web browser
~~~

这只移除了两个工具集，不代表剩下的只有 terminal 和 MCP。实际允许范围还要查看当前会话暴露的工具，MCP Server、插件和角色的工具集配置也要一起核对。


## SOUL.md：把角色要求写成可检查的工作方式

在各 Profile 的实际数据目录保存自己的 SOUL.md。下面是内容示意，写入哪个 Profile 应通过 profile show 确认。

~~~bash
hermes profile show researcher
hermes profile show writer
hermes profile show reviewer
~~~

researcher 的 SOUL.md：

~~~markdown
# Researcher

你负责技术调研，输出以下内容：
1. 核心结论与适用版本。
2. 每条关键事实的原始来源、链接和核查日期。
3. 已确认事实、社区反馈和未解决问题，分别标明。
4. 建议的文章方向与需要继续验证的点。

不要把二手转述当成官方保证。完成时记录产物位置。
~~~

writer 的 SOUL.md：

~~~markdown
# Writer

面向 AI 开发者，用中文写清楚问题、机制与使用边界。
以研究材料为依据，每个重要结论能追溯到来源。
代码用带语言标识的 Markdown 围栏，标出依赖和版本。
文章约 2000～3000 个汉字；复杂内容以清晰解释优先。
只修改分配给自己的稿件，不执行发布操作。
~~~

reviewer 的 SOUL.md：

~~~markdown
# Reviewer

检查技术事实、参数名、示例结构、资料日期和排版。
缺陷写明位置、原因与可操作修改要求。
审阅记录包含稿件路径与 SHA-256，避免批准后稿件被替换。
自动检查结论与人工批准分别记录，不自行宣称已获批准。
~~~

人格文件影响模型行为，但可靠的发布门槛还需要任务状态、文件版本与发布工具共同检查。


## Gateway：配置入口，再确定通知送到哪里

~~~bash
hermes gateway setup
hermes gateway run
~~~

先用前台日志检查连接，再按第 04 篇选择长期服务方式。不要一边保留前台进程，一边重复启动同一 Bot 的服务。

数据根目录 .env 中的配置示意：

~~~dotenv
TELEGRAM_BOT_TOKEN=REPLACE_WITH_REAL_BOT_TOKEN
TELEGRAM_ALLOWED_USERS=REPLACE_WITH_NUMERIC_USER_ID
~~~

| 检查项 | 要区分的情况 |
| --- | --- |
| Bot 连通 | Token 正确不等于用户已准入 |
| 消息路由 | 入口 Agent 使用哪一个 Profile |
| Kanban 通知 | 是否订阅这张任务，以及平台能否投递 |
| Cron 投递 | home channel 或显式 platform:chat_id 是否正确 |
| 后台运行 | Gateway 活着不等于每个工具后端都可用 |

有些部署用一个 multiplex Gateway 服务多个 Profile，有些只给协调角色配置消息入口。无论采取哪种形式，都要明确入口身份与实际任务的执行身份。


## GitHub MCP：有令牌之后，还要有连接与工具

只在 .env 中填写 GitHub Token，并不会自动增加 GitHub 工具。必须配置 MCP Server 的传输方式、认证和实际允许操作，再检查连接。

~~~bash
hermes -p publisher mcp list
hermes -p publisher mcp test github
~~~

github 是已配置 Server 的本地名称；如果实例中用了别的名字，测试命令也要换成那个名称。

| 对象 | 用途 | 保存在哪里 |
| --- | --- | --- |
| 仓库地址与文章目录 | 稳定的业务事实 | 项目说明、Profile 记忆或技能 |
| Token / 私钥 | 访问凭证 | 对应 Profile 的凭证配置 |
| MCP Server 定义 | 如何连接 GitHub 工具 | mcp_servers 配置 |
| 工具 Schema | 当前能做什么、参数是什么 | 连接后实际发现的工具 |

课件的 mcp_github_create_or_update_file 是某种 Server 的工具示意。不同 Server 的工具名、branch 参数和权限会变化；本文用 Git + gh 说明 PR 流程，MCP 方式按实际 Schema 做同样的分支与 PR 操作。

Provider 凭证与 GitHub 凭证用途不同，不能因为模型能回复，就认为 publisher 已能访问仓库。


## Memory：稳定事实按角色保存，文章材料明确交接

课件在一次聊天中让 Hermes 记住 Hugo、Markdown 和 PR 流程。当前使用不同 Profile 时，记忆通常按各自数据根目录保存，不会自动变成全团队共享记忆。

给 Writer 的首次说明可以写成：

~~~text
请在当前 writer Profile 中记住这些稳定约定：
博客使用 Hugo，文章放在 content/posts/，正文是 Markdown。
frontmatter 至少包含 title、date、tags、author，并按项目要求写 description。
中文技术文章保留必要英文术语，代码块写明语言。
发布由 publisher 提交 PR，合并后由仓库部署流程上线。
访问令牌不写入记忆或正文。仓库地址以项目配置为准。
~~~

| 内容 | 更适合的载体 |
| --- | --- |
| 中文偏好、文章格式 | 对应角色的 Memory / SOUL.md |
| 多角色共同遵守的规则 | 版本化项目说明或共享 Skill |
| 当前主题的资料与草稿 | 持久文件、Kanban 附件和任务交接 |
| 当前运行进度 | 任务状态、Comment、Run 与 Event |
| 访问凭证 | 凭证配置 |

模型是否实际写入记忆，应检查当前工具结果或文件，不要把“记住”这句输入当成写入成功的证据。上下文压缩也不能代替保存文章、审核记录和版本号。


## 持久工作目录：解决“上一角色的文件在哪里”

本例给每篇文章一个独立的持久目录，角色按任务依赖依次写入：

~~~text
/srv/blog-work/hermes-swarm/
├── brief.md
├── research/
│   ├── architecture.md
│   └── community.md
├── draft.md
├── review.md
├── approval.json
└── pr-body.md

/srv/tech-blog/
└── content/posts/hermes-swarm-deep-dive.md
~~~

| 文件 | 主要写入者 | 下游用途 |
| --- | --- | --- |
| brief.md | 选题与协调角色 | 确认范围、受众和完成标准 |
| research/ 下各文件 | 对应 researcher Worker | Writer 引用研究依据 |
| draft.md | Writer | Reviewer 审阅，Publisher 复制批准版本 |
| review.md | Reviewer | 保存核查结果与修改要求 |
| approval.json | 已授权的审批流程 | 绑定批准人与稿件摘要 |
| pr-body.md | Publisher | 说明文章变化与验证结果 |

dir workspace 要使用实际存在的绝对路径。共享目录并不解决并发冲突：两个 researcher 分别写两个文件，reviewer 读稿，writer 独占修改 draft.md。

课件多处使用 scratch。当前 scratch 在任务完成时会清理；需要跨角色交付时，应声明 artifacts 让系统保存附件，或采用持久 dir / worktree，并在摘要中记录真实路径。


## 创建 blog Board：队列和文件目录不是同一个东西

~~~bash
hermes kanban boards create blog --name "技术博客工作板" --switch
hermes kanban --board blog init
hermes kanban boards show
~~~

重复准备时先查看已有 Board，避免把“新建”当成幂等操作。init 用来初始化数据库，已存在时不会重复创建。

本篇后续示例都显式指定 --board blog 或工具参数 board=blog，减少当前 Board 切换造成的混淆。

| Board 保存 | Workspace 保存 |
| --- | --- |
| Task、Link、Comment 与运行记录 | 研究材料、稿件与审核文件 |
| 任务分配与进度 | 文件的实际内容 |
| 完成摘要和附件记录 | Git 工作目录中的变更 |

Board 的位置、各 Profile 的数据目录和执行时注入的 HERMES_KANBAN_BOARD / 数据库目标应一致。Worker 由 Dispatcher 传入正确的板信息，避免自己按另一个 Profile 的默认 Board 重新查任务。


## 内置拆解器与 Orchestrator Profile：配置名容易让人误会

Gateway 所用配置中的示意：

~~~yaml
kanban:
  dispatch_in_gateway: true
  dispatch_interval_seconds: 60
  auto_decompose: true
  orchestrator_profile: orchestrator
~~~

| 配置 | 当前含义 |
| --- | --- |
| dispatch_in_gateway | 由 Gateway 承载 Dispatcher |
| dispatch_interval_seconds | 后台检查与调度间隔 |
| auto_decompose | 是否自动拆解 triage 中的粗任务 |
| orchestrator_profile | 拆解后根协调任务归哪个 Profile |

当前官方文档明确：**orchestrator_profile 不会把该 Profile 的 SOUL.md、技能和自定义逻辑加载到内置拆解调用中**。内置拆解器使用自己的提示与 auxiliary.kanban_decomposer 模型配置，再参考 Profile 名册和描述生成任务图。

如果希望完全按自定义协调角色的逻辑建图，可以采用手动模式，由该角色显式创建任务。本例展示内置拆解的设计思路，但生成图仍需核查产物路径、审核门槛和发布完成标准。

设置了配置不等于 Worker 已经运行。Board 有卡片之后，还要确认 Gateway、Dispatcher、assignee、工具与目录都满足条件。


## Cron 选题流水线：时间、输入和最终输出都要写全

课件用三次定时运行组织素材：07:00 收集、07:30 筛选、08:00 生成简报。它们是不同会话，需要完整 prompt。

~~~yaml
timezone: Asia/Shanghai
~~~

以上为示例时区，实际还会受 HERMES_TIMEZONE 等配置影响。先确认当前 Profile 的有效时区，07:00 才有明确含义。

| 任务 | 输入 | 最终输出应包含 |
| --- | --- | --- |
| blog-news-collect | 指定公开资讯源 | batch_id、时间、标题、URL、摘要与访问失败项 |
| blog-topic-select | 收集任务已完成输出 | 沿用 batch_id、3 个选题、评分与去重依据 |
| blog-brief-generate | 筛选任务已完成输出 | 选题简报、提纲、参考资料、未确认问题 |

把选题价值定义清楚，例如技术深度、读者兴趣、与已有文章的区别。HN、知乎等来源有各自的访问条件；网页不可获取时记录缺失，不能编造新闻或来源链接。

本篇 JSON 使用当前模型工具 cronjob_manage。它们是工具参数示例，不是在系统终端中输入的 Python 函数调用。


## 创建采集任务：搜索工具与写作技能不要混在一起

~~~json
{
  "name": "cronjob_manage",
  "arguments": {
    "action": "create",
    "name": "blog-news-collect",
    "schedule": "0 7 * * *",
    "prompt": "按当前配置时区采集今天的 AI 与前端公开资讯。为每次运行生成 batch_id，输出 generated_at、资料日期、标题、原始 URL、简要说明及无法访问的来源。最终回复包含完整采集摘要，不仅报告文件路径；不创建文章任务或执行发布。",
    "enabled_toolsets": ["web"],
    "deliver": "local"
  }
}
~~~

课件把 web_search 放进 skills。web_search 是工具名，不是天然存在的一份 Skill；如果确实安装了选题技能，skills 应填那份技能的实际名称。

这里先用 local 表示保存输出。需要 Telegram 通知时，三个任务都应按部署目标设置 deliver，例如 telegram:CHAT_ID，其中 CHAT_ID 必须替换为实际目标。平台名本身通常依赖 home channel 配置，不能凭准入用户 ID 就假定投递地址已确定。

创建后记录工具返回的真实 job ID，先 list 已有任务，重试时优先 update，避免每天重复创建一套同名流水线。


## 创建筛选与简报任务：context_from 使用真实 ID 数组

~~~json
{
  "name": "cronjob_manage",
  "arguments": {
    "action": "create",
    "name": "blog-topic-select",
    "schedule": "30 7 * * *",
    "context_from": ["COLLECT_JOB_ID"],
    "prompt": "读取注入的采集结果，先确认 batch_id、日期和 generated_at 属于本轮。按技术深度、读者兴趣和与已有文章的差异选出最多 3 个主题，保留来源与评分理由。缺少当前批次资料时报告缺失，不用旧资料冒充今天；最终回复输出完整结果。",
    "deliver": "local"
  }
}
~~~

~~~json
{
  "name": "cronjob_manage",
  "arguments": {
    "action": "create",
    "name": "blog-brief-generate",
    "schedule": "0 8 * * *",
    "context_from": ["SELECT_JOB_ID"],
    "prompt": "确认注入的选题结果属于本轮，对选出的主题逐一生成简报：batch_id、目标读者、文章问题、提纲、核心观点、参考链接、预估篇幅和未确认问题。等待负责人选题，不直接发布或自动跳过审核。最终回复输出简报全文。",
    "deliver": "local"
  }
}
~~~

COLLECT_JOB_ID 和 SELECT_JOB_ID 是占位符，必须用前面成功创建或 list 返回的 ID。当前 context_from 的模型 Schema 是数组，单个上游也不能改成裸字符串。


## 三个时刻不是三道依赖锁

context_from 读取上游最近已完成输出，不会等待本轮上游结束。采集如果到 07:40 才完成，07:30 筛选可能读到昨天的数据。

| 情况 | 下游怎样处理 |
| --- | --- |
| 上游没有完成记录 | 报告缺失，停止生成当日结论 |
| 上游日期不是今天 | 标明过期，等待重跑或人工处理 |
| 上游失败或结果不完整 | 保留失败信息，避免把缺失当成无新闻 |
| 多份结果混入 | 逐份核对 batch_id，再合并 |
| 上游正文很长 | 关键日期、结论和 URL 放在最终输出前面 |

### 两种更强的顺序设计

可以把“采集 → 筛选 → 简报”放在一次 Cron 中顺序完成，或者让 Cron 只生成带批次号的任务，再用 Kanban 依赖组织后续工作。错开时间有用，但不能证明数据是同一批。

~~~bash
hermes cron status
hermes cron list
~~~

按创建任务的 Profile 查看状态与输出。普通 CLI 聊天窗口存在，不表示后台定时器一直在线。


## Telegram 发起文章：把模糊需求变成任务正文

负责人在会话中可以提出：

~~~text
请围绕 Hermes Agent Kanban Swarm 写一篇面向 AI 开发者的深度文章。
采用当前实际目标版本，同时标明课件版本差异。
覆盖任务依赖、Worker 调度、产物交接与常见误区。
先完成调研、草稿与技术审核，再等我批准具体稿件版本。
最终提交博客仓库 PR，并记录部署状态。
~~~

给消息入口的路由规则内容示意：

~~~markdown
## 博客任务路由

明确的文章创作请求进入 blog Board。
把主题、受众、资料版本、输出位置和完成标准写入 task body。
使用 kanban_create 创建任务，保存返回的真实 task_id。
同一 request_id 的重试使用相同 idempotency_key，避免重复建卡。
回复用户任务编号与当前状态，未观察到拆解成功时不宣称已开始写作。
普通咨询可以正常回答。
~~~

课件前文使用 Telegram，路由提示中又写微信。本例统一使用 Telegram；切换平台时，要同时修改入口配置与通知目标，不能只换提示里的名字。


## 创建文章根任务：明确工作目录、去重键与验收边界

~~~json
{
  "name": "kanban_create",
  "arguments": {
    "board": "blog",
    "title": "Hermes Kanban Swarm 深度文章：调研、审核与 PR 交付",
    "assignee": "orchestrator",
    "triage": true,
    "workspace_kind": "dir",
    "workspace_path": "/srv/blog-work/hermes-swarm",
    "idempotency_key": "blog:hermes-swarm:REQUEST_ID",
    "body": "面向 AI 开发者，写约 2000～3000 个汉字的中文技术文章。确认目标版本，调研架构与社区反馈，写入持久研究文件与 draft.md，完成技术审核，再等待人工批准对应 SHA-256。Publisher 提交文章 PR，记录 URL、提交与构建结果；合并和上线状态另行记录。拆解时为每个角色显式指定产物路径、前置依赖和完成标准。"
  }
}
~~~

REQUEST_ID 应来自这次实际请求，不能对所有文章固定使用一个常量。去重键负责避免同一次请求重试建重复卡，不负责判断两个相近主题是不是同一篇文章。

创建成功后看返回状态与实际图。根任务指定的目录也不能被当成所有子任务必然继承的保证；拆解结果中要逐项检查 workspace 与 body。


## 拆解结果：并行调研，再汇合写稿与审核

手动拆解示意：

~~~bash
hermes kanban --board blog decompose ROOT_TASK_ID
hermes kanban --board blog list
hermes kanban --board blog show ROOT_TASK_ID
~~~

ROOT_TASK_ID 来自 create 返回值。自动拆解已启用时，由后台 tick 处理，不需要再反复手动触发。

下面描述的是文章业务阶段的期望顺序，ROOT 与各叶子任务的实际关系由拆解器另行建立：

~~~text
研究 A：架构、源码、版本 ─┐
                        ├→ 写稿 W → 审核 R → 人工批准 H → PR 提交 P
研究 B：案例、反馈、限制 ─┘                                ↓
                                                   合并 / 部署 / 线上检查
~~~

| 业务任务 | assignee | 主要验收 |
| --- | --- | --- |
| A：技术调研 | researcher | architecture.md 带可核查来源 |
| B：社区调研 | researcher | community.md 区分事实与反馈 |
| W：写稿 | writer | 读取 A、B，形成完整 draft.md |
| R：技术审核 | reviewer | 记录缺陷、版本与审核结论 |
| H：批准门槛 | 已授权的人类审批流程 | 批准文件绑定本次稿件摘要 |
| P：提交 PR | publisher | 检查批准版本，提交指定文章 |

人工批准是这里的业务门槛，不是叫一个 Agent“human”就自动完成。它可以表现为待输入的阻塞卡或明确审批记录，但程序必须检查记录确实对应目标稿件。


## 依赖与修改循环：不要让发布卡抢在审批前运行

Link 的方向是上游 → 下游。Writer 必须等待两个研究任务；Publisher 必须等待技术审核和人工批准。

~~~bash
hermes kanban --board blog link RESEARCH_A_ID WRITER_ID
hermes kanban --board blog link RESEARCH_B_ID WRITER_ID
~~~

上面的 link 用于已有卡片的补充依赖，ID 必须先从 Board 获取。更稳妥的方式是在创建下游卡时用 parents 明确前置项，避免调度开始后才补锁。

| 问题 | 合理的处理 |
| --- | --- |
| 草稿需要补解释或修正代码 | 给 Writer 明确修改任务，完成后重新审核 |
| 当前审查使用原生 review 模式 | Reviewer 可 request_changes 返回实现者 |
| 需要负责人选择目标版本 | needs_input 阻塞，补决定后恢复 |
| 发布缺少访问权限 | capability 阻塞，不把重试当成修复 |
| 网络短暂失败 | transient，并按运行策略重试 |

如果采用独立 reviewer 卡，unblock reviewer 只会恢复这张卡，不会自动重新启动 writer。需要创建或恢复修改任务，并让复审等待修改完成。

课件的“审核卡 blocked → 人工 unblock → Writer 自动修改”省略了这层路由；本文明确写出修改、复审、重新批准几个动作。


## Swarm 是一种建图方式：先看拓扑是否适合文章流程

当前 CLI 的 Swarm v1 组织为：并行 Worker → verifier → synthesizer。

~~~bash
hermes kanban --board blog swarm "产出 Hermes Swarm 研究与综合初稿，保留来源；不执行发布" \
  --worker "researcher:架构、版本与源码核查" \
  --worker "researcher:公开使用案例与社区反馈" \
  --verifier reviewer \
  --synthesizer writer \
  --idempotency-key "blog:hermes-swarm:RESEARCH_REQUEST_ID"
~~~

这里 Reviewer 在 Writer 前面，核验的是研究结论，而不是最终文章。最终稿仍要再审阅；publisher 和人工批准门槛也不会因为 swarm 命令存在就自动附加。

| 方式 | 适合什么 |
| --- | --- |
| triage + decompose | 按主题和验收条件生成自定义任务图 |
| swarm | 固定的并行研究、核验、综合拓扑 |
| 手工创建卡与 parents | 第一次验证精确的阶段顺序与权限边界 |

同一文章通常选择其中一种作为起点，再扩展所需阶段。不要同时执行前面的根任务示例和本节 Swarm 示例，再把两套图误认为同一条流水线。


## 研究 Worker：开始时读任务，结束时留下可用材料

~~~json
{
  "name": "kanban_show",
  "arguments": {
    "board": "blog",
    "task_id": "RESEARCH_A_ID"
  }
}
~~~

researcher 先确认主题、目标版本、已做工作、输出文件和下游需求，再搜索官方文档、源码与发布说明。社区反馈需要保留原文出处，不能据此推断普遍效果。

研究文件内容示意：

~~~markdown
# Swarm 技术研究

目标版本：实际核查的版本或提交。
核查日期：本次研究的实际日期。

## 已确认事实
- CLI 的 Swarm v1：并行 Worker → verifier → synthesizer。
- 创建任务图与 Dispatcher 运行 Worker 是不同阶段。

## 来源
- 官方 Kanban 文档：记录具体章节和 URL。
- 对应版本源码：记录文件路径与提交链接。

## 待确认
- 部署版本是否包含本文采用的配置项。
~~~

研究结论只在适用版本下成立。在线文档、课件和 main 不一致时，先记录差异，再决定文章按哪个版本写。


## 一次调研中的委派：子任务要有完整目标

~~~json
{
  "name": "delegate_task",
  "arguments": {
    "tasks": [
      {
        "goal": "核对公开发布说明中 Kanban Swarm 的版本变化，返回版本、日期、原始链接和未确认项。",
        "context": "用于 Hermes 技术文章的事实核查。只读调研，不修改稿件或任务状态。返回结果由父 Worker 汇总保存。"
      },
      {
        "goal": "核对 GitHub 上 Swarm 的公开设计讨论与使用限制，区分已实现行为、计划和社区反馈。",
        "context": "与发布说明调研并行，保留具体 issue/PR 链接和核查日期。不要把讨论中的建议写成产品保证。"
      }
    ]
  }
}
~~~

课件在每个 tasks 元素中加入 toolsets。当前模型 Schema 不再公布这个参数，子工具范围由父配置和委派实现决定；因此本例仅使用当前合法字段，不用提示里的“只读”冒充工具级隔离。

Kanban 负责跨任务持久交接，delegate_task 负责当前 Worker 内的子问题。子 Agent 结束后，父 Worker 仍需核对、合并并保存最终研究结果，不能只转发一句“他们都完成了”。

层数、并发与结束条件沿用第 04 篇的配置原则。并行会放大模型和搜索调用，适合独立问题，不适合多人同时改同一份稿件。


## 研究完成摘要：保留事实、来源与产物路径

~~~json
{
  "name": "kanban_complete",
  "arguments": {
    "board": "blog",
    "task_id": "RESEARCH_A_ID",
    "summary": "完成架构与版本核查，研究材料已保存。核心结论、来源和待确认项供 Writer 使用。",
    "metadata": {
      "research_path": "/srv/blog-work/hermes-swarm/research/architecture.md",
      "target_version": "ACTUAL_VERSION_OR_COMMIT",
      "source_count": 3,
      "open_questions": ["确认实际部署版本与公开 main 的差异"]
    },
    "artifacts": ["/srv/blog-work/hermes-swarm/research/architecture.md"]
  }
}
~~~

字段里的路径、来源数量和版本应来自真实产物，这里只是参数结构示意。文件必须在完成时存在，不能先 complete 再去补写文件。

当使用 scratch 时，声明 artifacts 让系统在清理前保存文件；Writer 应从上游交接或附件列表取得实际保存路径，而不是依赖已清理的 scratch 原路径。

只在摘要里说“资料已找到”，Writer 无法知道找到什么。重点结论放在 summary / metadata，完整材料放到持久文件或附件，二者配合使用。


## Writer 的输入：研究材料、风格样本与当前角色历史

| 输入 | 能提供什么 | 不能假定什么 |
| --- | --- | --- |
| 两份研究产物 | 本篇的事实与参考链接 | 研究已覆盖全部技术细节 |
| 写作 Skill | 可复用的文章流程 | 已安装后一定自动加载 |
| session_search | 当前作用域的历史讨论 | 能跨所有 Profile 搜索 |
| 文件引用或 read_file | 实际风格样本 | 相对路径在任何 Worker 都能解析 |
| Memory / SOUL.md | 稳定格式与语气要求 | 能直接替代具体稿件资料 |

示例任务输入：

~~~text
读取 brief.md 与 research/architecture.md、research/community.md。
参考项目中已确认的写作样本，先给出提纲，再写入 draft.md。
引用结论标出来源，尚未验证的行为写成待确认项。
不更改研究文件，不发布到 GitHub。
~~~

课件写 @file:content/posts/...。文件引用语法依入口和版本而异，不把它当成所有 API / Worker 都支持的统一协议。交互式入口用实际支持的引用方式，后台 Worker 则通过明确路径和读取工具获取文件。

查到旧会话只表示存在历史讨论；当前写作样本应优先使用确认过的文件版本。


## 文章模板：frontmatter 和正文都保持可检查

draft.md 内容示意。外层用波浪线代码块展示整篇 Markdown，里面的 Python 用反引号，避免嵌套围栏提前结束。

~~~markdown
---
title: "Hermes Agent Kanban Swarm：从并行研究到文章交付"
date: "2026-10-09T10:00:00+08:00"
tags: ["Hermes Agent", "Kanban", "Swarm"]
author: "技术团队"
description: "解释任务依赖、Worker 调度、产物交接与博客发布流程。"
draft: true
---

# 为什么多 Agent 完成了任务，文章仍然没交付？

从一次具体的研究与写稿场景切入，说明任务状态与文件产物的关系。

## 架构与流程

说明 Gateway、Dispatcher、Profile、Task 和 Workspace 怎样配合。

## 示例

```python
def choose_topics(items, limit=3):
    if limit < 1:
        raise ValueError("limit must be positive")
    return items[:limit]
```

## 验证与边界

说明示例适用版本、执行前提与未验证行为。

## 参考资料

列出实际核查过的官方文档与源码链接。
~~~

date 取真实发布日期；draft: true 适合草稿。批准发布时，draft 标志、发布日期与内容的修改都会影响文件摘要，应以最终拟发布文件重新核对和批准。

description 不在课件最初四个字段中，但后面的 SEO 检查需要它。本篇把字段需求提前统一，避免审核阶段临时要求另一个模板。


## 技术审核：格式、事实与构建是不同检查

| 检查 | 常见问题 | 记录什么 |
| --- | --- | --- |
| frontmatter | YAML 错误、字段缺失、日期不合法 | 文件、字段与解析结果 |
| API 与命令 | 名称错误、参数过期、占位符未说明 | 目标版本与核查来源 |
| 代码块 | 围栏未闭合、语言不明确、示例缺步骤 | 对应章节与修改要求 |
| 正文逻辑 | 用推测支持结论、遗漏限制 | 具体位置与依据 |
| 文章引用 | 链接失效、二手资料被当成官方结论 | 实际访问情况 |
| Hugo 构建 | 主题依赖、模板错误、草稿未渲染 | 命令、退出状态与构建日志 |
| 发布版本 | 审核后文件变化 | SHA-256 与重新审核记录 |

课件的 Post-Write Linting 可以作为快速反馈，但是否存在、检查哪些格式取决于当前安装版本与相关工具。写完后出现“未报错”，不能当作全部文章质量已经验证。

运行 Python / Shell 示例、执行构建和静态语法检查属于不同验证。文章中应准确标出做过哪一种，不能把静态检查写成“实际运行成功”。


## 人工审批：批准哪一份稿件，比“通过”两个字更重要

审核记录应保存最终拟发布稿件的路径、摘要、批准人和批准时间。下面是业务记录示意，不是 Hermes 内置审批格式：

~~~json
{
  "request_id": "ACTUAL_REQUEST_ID",
  "draft_path": "/srv/blog-work/hermes-swarm/draft.md",
  "draft_sha256": "SHA256_OF_APPROVED_FILE",
  "technical_review": "passed",
  "approved_by": "ACTUAL_AUTHORIZED_REVIEWER",
  "approved_at": "ACTUAL_APPROVAL_TIME",
  "permission": "submit_pr"
}
~~~

| 稿件状态 | 接下来怎么做 |
| --- | --- |
| 技术审核未通过 | 修改与复审 |
| 技术审核通过，尚无人批准 | 等待明确批准 |
| 已批准但文件摘要变化 | 重新审核并更新批准记录 |
| 批准范围为 submit_pr | 提交 PR，不据此自动推断合并权限 |
| PR 已合并 | 查看部署运行与线上结果 |

审批文件要来自实际授权流程，不能由 Writer 自己填一份 approved_by 后宣布获批。Publisher 应对照实际文件计算摘要；仅在提示词中要求检查，仍可能被模型遗漏，需要执行流程中的程序校验配合。

人工输入补齐时，在任务评论中记录决定，再恢复相应任务。每次恢复先重读任务与评论，避免沿用旧结论。


## 审查退回与外部阻塞：使用对应状态

在原生 review 流程中，Reviewer 从 review 列认领任务后，可以退回实现者：

~~~json
{
  "name": "kanban_request_changes",
  "arguments": {
    "board": "blog",
    "task_id": "DRAFT_TASK_ID",
    "reason": "第 3 节把 context_from 写成单字符串；请改为数组，并解释上游本轮未完成时可能读取旧资料。更新后重新检查稿件摘要。"
  }
}
~~~

这与独立 reviewer 卡的复审路径不同，不能对任意运行中的卡片直接套用该工具。

如果确实缺少人工决定，则使用阻塞：

~~~json
{
  "name": "kanban_block",
  "arguments": {
    "board": "blog",
    "task_id": "APPROVAL_TASK_ID",
    "kind": "needs_input",
    "reason": "技术审阅已完成，请负责人确认记录中对应 SHA-256 的最终稿件，授权提交 PR。尚未获批准。"
  }
}
~~~

通知是否发送取决于任务订阅与 Gateway 状态。看到 blocked 状态，不应直接宣称负责人已收到通知。反复 unblock 而不补前置条件，只会把同一个问题重新送回队列。


## Publisher 准备：先核对批准稿，再检查仓库状态

发布任务开始时读取最终稿、审核记录、批准记录和目标仓库配置。

~~~bash
cd /srv/tech-blog
git status --short
git branch --show-current
sha256sum /srv/blog-work/hermes-swarm/draft.md
~~~

这是发布准备示例，返回结果需要与 approval.json 对照。摘要不一致时停止提交并重新审核。

| 条件 | 处理方式 |
| --- | --- |
| 仓库已有无关修改 | 先明确归属，避免把它们加入文章提交 |
| 正在错误分支 | 切换到本次约定的文章分支 |
| 同一请求已有分支或 PR | 更新既有 PR，避免重复创建 |
| 凭证或分支权限不足 | 记录 capability 阻塞 |
| 批准后改动 draft / date / frontmatter | 重新计算摘要并确认批准 |

课件每次都在 workspace 中 clone，再复制 /path/to/draft.md，隐藏了稿件从哪个任务来的问题。本篇采用固定博客工作目录和明确产物路径，也可以使用独立 Git worktree，但需要把分支与文件交接写全。


## 提交文章 PR：复制单篇文章，检查变更，再提交

下面是首次创建文章分支的流程示意，假定仓库和批准稿件已准备好，且当前工作目录无冲突。它不负责自动完成前面的业务审批校验。

~~~bash
cd /srv/tech-blog
git switch -c blog/hermes-swarm-deep-dive
cp /srv/blog-work/hermes-swarm/draft.md content/posts/hermes-swarm-deep-dive.md
git add content/posts/hermes-swarm-deep-dive.md
git diff --cached --check
git diff --cached -- content/posts/hermes-swarm-deep-dive.md
~~~

先审阅暂存差异，确认只包含这篇文章，frontmatter 符合发布要求。再按博客项目已验证的方式构建：

~~~bash
hugo --minify --destination /srv/blog-work/hermes-swarm/build-preview
~~~

Hugo 默认可能不渲染 draft: true 的文章，因此真正发布前应确认最终稿已采用项目允许的发布状态。预览草稿可以使用项目认可的草稿构建方式；普通构建退出成功不能证明这篇草稿已经输出页面。

确认变更和构建后，提交与创建 PR：

~~~bash
git commit -m "Add Hermes Kanban Swarm article"
git push -u origin blog/hermes-swarm-deep-dive
gh pr create \
  --base main \
  --head blog/hermes-swarm-deep-dive \
  --title "Add Hermes Kanban Swarm article" \
  --body-file /srv/blog-work/hermes-swarm/pr-body.md
~~~

新建分支示例只适合首次执行，重试时先检查分支和现有 PR。构建命令、主题依赖和 GitHub Actions 流程按实际博客仓库调整。


## PR 说明与 Kanban 交接：准确写“已提交”，再跟踪上线

pr-body.md 内容示意：

~~~markdown
新增 Hermes Kanban Swarm 技术文章，解释任务依赖、调度与产物交接。

验证记录：
- 目标版本和关键参数已对照实际资料。
- frontmatter、代码围栏和引用已检查。
- Hugo 构建结果按本次真实输出填写。
- 批准记录对应最终提交稿件的 SHA-256。

发布状态：PR 待审核与合并，部署及线上核验尚未完成。
~~~

提交 PR 任务的完成记录示意：

~~~json
{
  "name": "kanban_complete",
  "arguments": {
    "board": "blog",
    "task_id": "PUBLISHER_PR_TASK_ID",
    "summary": "已提交文章 PR，等待仓库审核与合并。尚未确认 GitHub Pages 上线。",
    "metadata": {
      "pr_url": "ACTUAL_PR_URL",
      "commit": "ACTUAL_COMMIT_SHA",
      "draft_sha256": "ACTUAL_APPROVED_SHA256",
      "deployment_status": "pending_merge"
    }
  }
}
~~~

| 事件 | 是否等于上线成功 |
| --- | --- |
| Git push 成功 | 只说明提交已进入远端分支 |
| PR 创建成功 | 只说明变更已进入审核入口 |
| PR 合并 | 还要等待部署流程 |
| Actions 构建成功 | 还需确认对应环境的发布结果 |
| Pages 部署成功 | 再核验目标 URL、正文与提交版本 |

根协调任务按自己的验收标准检查全部阶段。发布失败时保留 PR、提交与运行链接，下一次恢复从真实状态接续。


## Skills 沉淀：把反复验证的方法留下来

成功发布后，可以把选题、核查、写作和审核中稳定有效的步骤保存为写作 Skill。当前 skill_manage 的模型接口采用 operations 数组。

~~~json
{
  "name": "skill_manage",
  "arguments": {
    "operations": [
      {
        "action": "create",
        "name": "technical-blog",
        "category": "writing",
        "content": "---\nname: technical-blog\ndescription: 根据带来源的研究材料编写和审核中文技术博客，适用于需要可追溯引用的技术文章。\n---\n\n# 技术博客流程\n\n1. 确认目标版本、受众和选题价值。\n2. 读取研究材料，保留事实与来源映射。\n3. 用带语言标识的代码围栏写示例，说明依赖。\n4. 检查 frontmatter、引用和构建结果。\n5. 审核最终稿件摘要，批准后交由发布角色提交 PR。\n6. 分别记录 PR、部署与线上核验结果。\n"
      }
    ]
  }
}
~~~

这里只展示首次创建。已有同名技能时先读取，再局部 patch，避免覆盖之前验证过的内容。实际保存在哪个 Profile、其他角色怎样加载，要另行指定。

课件说“一次成功后后台 review agent 会自动创建技能”。这种行为受版本、配置与运行路径影响，不能把读者反馈好直接当成自动落盘的证据。应检查技能文件、provenance 和记录，再决定是否复用。


## Curator 与 Memory：维护方法，更新事实

~~~bash
hermes -p writer curator status
~~~

| 对象 | 维护动作 | 判断依据 |
| --- | --- | --- |
| 重复写作技能 | 审阅后合并或调整边界 | 工作流程是否重复，是否仍适用 |
| 长期未用技能 | 标记状态或归档 | 实际策略、管理范围与保护规则 |
| 写作长度偏好 | 更新对应 Profile 记忆 | 足够反馈与真实阅读效果 |
| 博客版本与主题 | 更新项目事实 | 仓库的实际配置 |

课件示例说 Curator 一周后合并两个 Skill，并将 30 天未用的技能标记 stale。当前常规维护默认以规则为主，模型 consolidate 需要相应配置；30 天也应按实际策略理解，不能保证这个案例运行一周就出现相同结果。

查看本次状态给出的真实日志目录，再读报告，不硬编码课件日期的 REPORT.md 路径。

“文章 2000～2500 字效果最好”需要真实反馈支持；Memory 是可维护的偏好和事实，不是把每次聊天中的猜测自动变成可靠统计。多角色共享内容仍通过项目文件或明确共享目录管理。


## Dashboard 与 TUI：看管理状态，也看具体运行

~~~bash
hermes dashboard --host 127.0.0.1 --port 9119 --no-open
hermes kanban --board blog watch
hermes kanban --board blog stats
~~~

当前 Dashboard 默认端口为 9119。本例保留这个端口，API Server 后面用 8642，避免两个服务抢占课件示例中的 8080。

| 想看什么 | 优先入口 |
| --- | --- |
| 卡片、依赖、状态和交接 | Dashboard Kanban 或 kanban show |
| 调度与运行进度 | kanban watch、Event 与 Worker 日志 |
| 当前管理的 Profile 配置 | Dashboard 的 Profile 选择器 |
| 当前交互会话的工具过程 | 对应 CLI / TUI |
| 模型与任务消耗 | 当前运行记录与相应计量日志 |

~~~bash
hermes --tui
~~~

另开一个 TUI 不会自动镜像 Telegram 会话或其他进程 Worker 的实时过程。应查看实际会话或对应任务日志，不能把一个新窗口当成“全系统观察器”。

当前 Dashboard 可以管理多个 Profile，切换前后确认页面指向的身份。看得到任务不代表后台正在派发，读到日志也不代表产物已经验收。


## No-Agent 健康检查：检查页面，避免固定月份误判

课件通过首页有没有“2026-06”判断最新文章是否正常。站点进入新月份、首页分页或修改模板后，这个判断会失效。本例分别检查首页与目标文章，并校验目标正文标记。

脚本保存为当前 Profile 数据根目录下 scripts/blog-health-check.py。配置示意：

~~~dotenv
BLOG_HOME_URL=https://TEAM.github.io/tech-blog/
BLOG_ARTICLE_URL=https://TEAM.github.io/tech-blog/posts/hermes-swarm-deep-dive/
BLOG_ARTICLE_MARKER=Hermes Agent Kanban Swarm
~~~

TEAM、文章路径和 marker 要换成实际发布地址与正文中可见的唯一标记。marker 只能证明找到了这段文本；要求确认最新部署版本时，应另用页面版本标记或部署记录核对提交。

~~~python
import os
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


def fetch_page(url):
    request = Request(url, headers={"User-Agent": "blog-health-check/1.0"})
    with urlopen(request, timeout=10) as response:
        if response.status != 200:
            raise ValueError(f"HTTP {response.status}")
        limit = 2_000_000
        body = response.read(limit + 1)
        if len(body) > limit:
            raise ValueError("page exceeds inspection limit")
        charset = response.headers.get_content_charset() or "utf-8"
        return body.decode(charset, errors="replace")


def main():
    home = os.environ.get("BLOG_HOME_URL", "").strip()
    article = os.environ.get("BLOG_ARTICLE_URL", "").strip()
    marker = os.environ.get("BLOG_ARTICLE_MARKER", "").strip()
    if not home or not article or not marker:
        print("BLOG ALERT: missing page URLs or article marker")
        return
    if not all(url.startswith("https://") for url in (home, article)):
        print("BLOG ALERT: configured page URLs must use HTTPS")
        return
    try:
        fetch_page(home)
        body = fetch_page(article)
        if marker not in body:
            print("BLOG ALERT: target article marker missing")
            return
    except (HTTPError, URLError, TimeoutError, OSError, ValueError) as exc:
        print(f"BLOG ALERT: page check failed ({type(exc).__name__})")
        return
    print("OK: homepage and target article passed the configured checks")


if __name__ == "__main__":
    main()
~~~

探测失败被转换为明确的业务告警，脚本正常结束；意外崩溃与调度失败仍看 Cron 运行状态。示例不记录凭证或完整错误 URL，也未进行任何网络探测。


## 给健康检查设置 Cron：脚本输出与投递是两步

~~~bash
hermes cron create "every 30m" \
  --no-agent \
  --script blog-health-check.py \
  --deliver local \
  --name blog-health-check
~~~

需要通知时，改为本实例已配置的实际 Telegram 目标。任务属于创建它的 Profile，该 Profile 必须能读到 scripts/ 下的脚本及相应环境变量。

~~~yaml
cron:
  script_timeout_seconds: 60
~~~

60 秒是本例主动选择的脚本上限，不是声称默认值为 60。页面请求还有自己的超时，Cron 超时负责限制脚本整体运行。

| 结果 | 含义 |
| --- | --- |
| no_agent=true | 调度器运行脚本，不调用 LLM |
| stdout 有内容 | 按任务投递规则发送或本地保存 |
| stdout 为空 | 当前模式可以静默 |
| 脚本异常结束 | 查看运行错误与失败通知配置 |
| 页面返回 200 | 只说明请求成功，还需正文与版本核验 |

本例成功时也输出 OK，便于第一次理解机制。长期使用可以令健康时不输出，故障与恢复时再输出；如果保存前次状态，状态文件应位于自己的持久目录，避免每次重启都误报。


## Hooks：调用次数与 Token 使用量需要不同事件

原课件的 token-tracker.sh 只记录“web_search called”，没有读取使用量，因此它统计的是调用次数。

| 要统计的内容 | 合适的事件或数据 |
| --- | --- |
| web_search 调用了几次 | post_tool_call，过滤 tool_name |
| 主 Agent 模型请求使用量 | post_api_request 的 usage |
| 上下文压缩等辅助模型调用 | 当前版本的 post_auxiliary_call |
| 搜索服务费用 | 搜索服务自己的计费记录 |
| 总成本 | 用量、模型单价、缓存规则与工具费用共同计算 |

工具返回的文本 Token 与一次模型 API 请求的账单用量不是同一个概念。Provider 可能省略部分字段，未知应标记 unknown，不能用 0 代替。

当前 Shell Hook 输入有 hook_event_name、tool_name、session_id、profile 和 extra。事件特有的 usage 等字段位于 extra，具体形状按实际版本核对。


## Hook 脚本示例：只写必要指标，stdout 保持协议 JSON

保存为相应 Profile 数据根目录下 agent-hooks/blog-metrics.py；配置中的路径换成真实绝对路径。

~~~python
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import sys


def main():
    payload = json.load(sys.stdin)
    event = payload.get("hook_event_name")
    extra = payload.get("extra") or {}
    record = {
        "at": datetime.now(timezone.utc).isoformat(),
        "event": event,
        "profile": payload.get("profile", "unknown"),
        "session_id": payload.get("session_id", ""),
    }
    if event == "post_tool_call":
        if payload.get("tool_name") != "web_search":
            return
        record.update({"tool": "web_search", "calls": 1})
    elif event in {"post_api_request", "post_auxiliary_call"}:
        usage = extra.get("usage")
        allowed = ("prompt_tokens", "completion_tokens", "total_tokens",
                   "input_tokens", "output_tokens")
        record["usage"] = {
            key: usage[key] for key in allowed
            if isinstance(usage, dict) and key in usage
        }
        record["usage_status"] = "reported" if record["usage"] else "unknown"
        record["api_request_id"] = str(extra.get("api_request_id", ""))
    else:
        return
    home = Path(os.environ["HERMES_HOME"])
    target = home / "logs" / "blog-metrics.jsonl"
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"blog-metrics: {type(exc).__name__}", file=sys.stderr)
    print("{}")
~~~

本例保存事件、关联 ID 和顶层使用量，不写原始 prompt、tool_input 或完整结果。若 Provider 把 usage 序列化成非字典对象，记录 unknown 后再适配实际字段，不凭空填数。

JSONL 在单进程下便于阅读。大量并发 Worker 写同一日志时，应改成按会话分文件或使用可并发的日志收集方式，再用关联 ID 去重汇总。


## 注册 Hook：在实际执行角色里加载

对应 Profile 的 config.yaml 片段：

~~~yaml
hooks:
  post_tool_call:
    - matcher: "^web_search$"
      command: "python3 /ABSOLUTE/PROFILE_HOME/agent-hooks/blog-metrics.py"
      timeout: 5
  post_api_request:
    - command: "python3 /ABSOLUTE/PROFILE_HOME/agent-hooks/blog-metrics.py"
      timeout: 5
  post_auxiliary_call:
    - command: "python3 /ABSOLUTE/PROFILE_HOME/agent-hooks/blog-metrics.py"
      timeout: 5
~~~

本例按 Linux / WSL 显式用 python3 调用脚本，不依赖 shebang 或可执行位。Windows 原生应改为实际 Python 解释器与脚本路径；路径有空格时，在 command 的参数内正确加引号。解释器与脚本读取权限按实际环境核对。

| 配置点 | 说明 |
| --- | --- |
| matcher | 工具事件按实际工具名过滤 |
| timeout | 限制本次 Hook 子进程等待 |
| stdout | 保持 JSON 协议输出 |
| stderr | 供排错的简短诊断 |
| 加载范围 | 各执行 Profile 配置，不是改协调角色就覆盖所有 Worker |

Shell Hook 的首次信任、插件 Hook 的加载和 Gateway 生命周期 Hook 是不同入口，参照第 03 篇确认。本文只展示配置与静态语法，没有安装或触发任何 Hook。


## SEO 插件：把“元数据检查”作为一项辅助审阅

课件通过一个插件检查标题、description 和正文。本篇把它收敛为元数据启发式检查：标题不能为空，描述应有内容，正文需要存在，过长标题给出提示。

原文的英文空格分词与“标题第一个词的密度”不适合直接评价中文技术文章，也不能证明 SEO 排名。长度只是提示，规则应按实际语言和项目需求调整。

目录示意：

~~~text
<REVIEWER_HOME>/plugins/seo-checker/
├── plugin.yaml
└── __init__.py
~~~

plugin.yaml：

~~~yaml
name: seo-checker
version: "1.0"
description: 检查技术博客标题、描述与正文是否满足基础约定
provides_tools:
  - check_seo
~~~

这是在课件插件基础上调整的教学示例，不是安装了某个已有的官方 SEO 服务。


## 插件实现：Schema 给模型看，handler 返回检查结果

__init__.py：

~~~python
import json


def register(ctx):
    schema = {
        "name": "check_seo",
        "description": "Check basic blog metadata; heuristic feedback only.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string", "description": "Article title"},
                "description": {"type": "string", "description": "Meta description"},
                "content": {"type": "string", "description": "Article body"},
            },
            "required": ["title", "description", "content"],
        },
    }

    def handle_seo(params, **kwargs):
        values = {}
        for key in ("title", "description", "content"):
            value = params.get(key)
            if not isinstance(value, str):
                return json.dumps({"ok": False, "error": f"{key} must be a string"})
            values[key] = value.strip()
        issues, warnings = [], []
        if not values["title"]:
            issues.append("标题为空")
        if not values["description"]:
            issues.append("description 为空")
        if not values["content"]:
            issues.append("正文为空")
        if len(values["title"]) > 70:
            warnings.append("标题超过本示例的 70 字符提示阈值，请人工检查展示效果")
        return json.dumps(
            {
                "ok": not issues,
                "issues": issues,
                "warnings": warnings,
                "scope": "metadata heuristic; not a ranking prediction",
            },
            ensure_ascii=False,
        )

    ctx.register_tool(
        name="check_seo",
        toolset="blog_tools",
        schema=schema,
        handler=handle_seo,
    )
~~~

handler 返回 JSON 字符串，既检查参数类型，也说明结果范围。70 是教学提示阈值，不是搜索引擎的普适限制；中文字符、显示宽度和移动端截断不能只用 len 精确判断。

这个工具不解析 Markdown frontmatter，不构建 Hugo，也不核查文章事实，相关检查仍由 Reviewer 的其他步骤完成。


## 启用插件：能看见工具之后，审核流程还要调用它

~~~bash
hermes -p reviewer plugins enable seo-checker
hermes -p reviewer plugins list
~~~

在 Reviewer 任务 body 或写作审阅 Skill 中明确要求：读取实际稿件的 title、description、正文，调用已加载的 check_seo，把结果写入 review.md。

| 情况 | 应检查什么 |
| --- | --- |
| 插件启用却没有工具 | register 是否成功，当前会话是否加载，工具集是否允许 |
| 工具结果通过，但构建失败 | 元数据启发式检查未覆盖工程构建 |
| reviewer 未调用工具 | 任务说明、Skill 与实际可用 Schema |
| 只在 default 启用了插件 | reviewer Profile 是否独立配置 |

课件说启用后 Reviewer 会自动调用。启用使工具有机会可用，是否调用仍由工作流与模型决定；要把必要检查写进明确的验收步骤，并检查真实工具结果。


## API Server：按当前入口启动，端口与认证明确配置

课件使用 hermes api start --port 8080。所核对的当前官方文档通过 Gateway 的 API Server 适配器启用 HTTP 入口，示例如下：

~~~dotenv
API_SERVER_ENABLED=true
API_SERVER_HOST=127.0.0.1
API_SERVER_PORT=8642
API_SERVER_KEY=REPLACE_WITH_A_STRONG_LOCAL_API_KEY
~~~

~~~bash
hermes gateway run
~~~

实际部署已有 Gateway 时应更新既有入口配置，不是再次启动一个争用消息凭证的实例。监听地址、端口和 key 还要按有效配置与日志核对。

| 服务 | 本篇示例端口 | 用途 |
| --- | --- | --- |
| Dashboard | 9119 | 人工管理、查看配置和任务 |
| API Server | 8642 | CMS / 外部程序提交 Agent 请求 |

API 默认请求仍由 Hermes 调用工具执行。接口返回一次文本回复，不自动保证创建了 Kanban 任务；需要明确路由，并核查返回的真实任务 ID。


## 外部 CMS 提交需求：带认证，把紧急程度与审核规则写清楚

请求体 urgent-request.json 示例：

~~~json
{
  "model": "hermes-agent",
  "messages": [
    {
      "role": "user",
      "content": "request_id=ACTUAL_CMS_REQUEST_ID。请把 content/posts/breaking-news.md 作为紧急文章需求进入 blog Board，先核查实际文件与目标版本，完成技术审阅及人工批准，再提交 PR。请返回实际任务编号，不要把紧急标记解释为已获发布许可。"
    }
  ],
  "stream": false
}
~~~

调用示意，终端中需已设置实际 API_SERVER_KEY：

~~~bash
curl --fail-with-body http://127.0.0.1:8642/v1/chat/completions \
  -H "Authorization: Bearer $API_SERVER_KEY" \
  -H "Content-Type: application/json" \
  -H "X-Hermes-Session-Id: cms-blog-request" \
  --data-binary @urgent-request.json
~~~

| 字段 | 作用 |
| --- | --- |
| Bearer key | 请求认证 |
| messages | 本次任务上下文 |
| X-Hermes-Session-Id | 请求关联的会话标识，不是业务去重键 |
| request_id | 业务请求身份，路由层据此生成 idempotency_key |
| 返回的 task_id | 核查真实建卡与后续进度 |

当前 Chat Completions 主要由客户端提供 messages 历史；单加会话标识不能替代完整对话输入。API 请求中的相对稿件路径也要按实际执行目录解析。

原课件让紧急请求“跳过常规审核”，但没有定义谁有这项权限或如何验证。这里保留紧急入口，采用明确的审阅与批准条件；如果业务确有紧急发布机制，应另定义可检查的授权规则，而不是只靠一句 prompt。


## 失败恢复：从真实状态继续，避免重复文章与重复 PR

| 故障位置 | 先核对 | 恢复动作 |
| --- | --- | --- |
| Cron 无输出 | Profile、时区、调度器、暂停状态 | 恢复对应任务，不重建整套流水线 |
| 筛选拿到旧新闻 | batch_id、日期、上游完成时间 | 等本轮完成后重跑下游 |
| Worker 无法读取材料 | workspace、持久路径、附件记录 | 修正交接与 body |
| 多人覆盖 draft.md | 文件归属与并发任务 | 分开产物，恢复单写者与依赖 |
| 审核退回后 Writer 不动 | 独立卡还是原生 review 模式 | 明确创建修改与复审工作 |
| 已批准稿又变化 | 文件 SHA-256 | 重新审阅与批准 |
| push / 创建 PR 失败 | 分支、现有 PR、访问权限 | 更新同一请求的既有状态 |
| 合并后部署失败 | Actions 日志与目标提交 | 修复构建或部署，不重复宣称上线 |
| 200 页面却无新正文 | URL、draft、缓存、部署版本 | 核验真实文章与版本 |
| Hook 无用量 | 事件、payload、Provider usage | 记录 unknown，按实际字段适配 |

恢复前先看任务、评论、运行记录与产物，再做下一步。对共享仓库、活跃 Worker 和现有 PR，不能只根据昨天的聊天摘要重新执行完整脚本。

业务 request_id、Kanban idempotency_key、文章路径、分支名与 PR URL 应能互相对应，这样重试时才知道是否已经完成了某一步。


## 按阶段验收：先证明一篇能走通，再加自动化

以下是学习时的验证顺序，本篇未执行这些实战动作。

| 阶段 | 验收目标 |
| --- | --- |
| 单角色准备 | Profile 能读取自己的配置、工具与项目规则 |
| 人工选题 | 一份简报能说明读者、问题与资料范围 |
| 调研交接 | 两份材料保存且 Writer 能实际读取 |
| 写作审阅 | 最终稿、缺陷处理与摘要记录完整 |
| PR 交付 | 批准稿进入独立分支，变更范围可审查 |
| 上线确认 | 合并、部署与目标页面对应同一版本 |
| 持续维护 | Cron、Hook 和检查脚本结果可追踪 |

第一次先走一个受控主题。流程稳定后再增加并行研究、Swarm、每日选题和外部 API，比较时间、费用与人工修订量，而不是只数启用了多少功能。

最小完整结果是一篇有来源、有批准版本、有 PR 和线上核验记录的文章。自动化越多，状态与产物之间的对应关系越需要明确。


## 课件与本篇写法差异速查

| 课件写法或结论 | 本篇处理 |
| --- | --- |
| Profile wrapper 短命令 | 用 hermes -p NAME 明确身份 |
| 所有角色共享 Memory / Skills | 说明默认按 Profile 保存，共享需额外配置 |
| web_search 放进 skills | 区分工具与技能，采集用 web 工具集 |
| cronjob + 字符串 context_from | 当前使用 cronjob_manage + ID 数组 |
| 三个 Cron 自动形成依赖 | 校验批次；context_from 只读已有输出 |
| orchestrator_profile 自动使用角色逻辑拆解 | 标明内置拆解器与根任务身份不同 |
| 普通 kanban_create / decompose 就是 Swarm | 区分建图入口与固定 Swarm 拓扑 |
| 委派 tasks 含 toolsets | 当前模型示例只用 Schema 公布字段 |
| scratch 草稿直接给后续读取 | 用持久目录或 artifacts 保存交付 |
| unblock reviewer 自动让 writer 修改 | 明确修改任务、复审或原生 request_changes |
| PR 流程与 push main 并存 | 统一为文章分支与 PR，合并后跟踪部署 |
| Git push 后即“已发布” | 分别记录 PR、合并、部署和线上核验 |
| Token Hook 只记录工具名称 | 标明调用次数与 API usage 是两种指标 |
| 固定月份判断新文章 | 检查目标页面、正文标记和部署版本 |
| 英文词密度判断中文 SEO | 改为元数据启发式提示，标明范围 |
| 启用插件后必然自动调用 | 要求任务流程显式调用并记录实际结果 |
| TUI 新窗口观察所有 Agent | 查看真实会话、任务事件与 Worker 日志 |
| hermes api start，和 Dashboard 都占 8080 | 当前 Gateway 启用 API，示例分别用 8642 / 9119 |
| 紧急请求直接跳过审核 | 给出明确审批与稿件版本条件 |

这些是本次对照公开接口后的整理，不代表在用户已安装实例上跑过整个博客系统。在线文档与 main 会变化，实际复用应核对安装版本的帮助、Schema 与日志。


## 本章复习：入口、依赖、产物、批准与上线

| 问题 | 本篇的关键判断 |
| --- | --- |
| Cron 与 Kanban 怎样配合？ | Cron 管时间，Kanban 管任务依赖与交接 |
| Profile 能否代替共享目录？ | Profile 保存身份状态，产物仍需明确位置 |
| 两个研究任务怎样进入 Writer？ | 前置任务完成 + 可读取材料 + 交接摘要 |
| Swarm 是否直接等于完整博客流程？ | 要查看拓扑，再补最终稿审核与发布阶段 |
| 什么才算人工批准？ | 授权人批准具体文件版本与动作范围 |
| 工具启用是否等于已检查？ | 还要看到实际调用与结果 |
| PR 创建是否等于发布？ | 仍需合并、部署与线上核验 |

可以尝试回答：

1. 07:30 筛选为什么可能拿到昨天的资讯？怎样证明资料属于本轮？
2. Writer 怎样读取另一个 Profile 的研究产物？为什么仅靠 session_search 不够？
3. Reviewer 阻塞后，怎样让修改工作真正回到 Writer？
4. 批准后只改了 date 或 draft 标志，为什么仍要检查批准版本？
5. 为什么课件的 web_search 日志不能直接算出总 Token 费用？
6. 页面返回 200 后，还需要哪些证据才能确认新文章已上线？

练习时先画出一篇文章的业务阶段图，再逐个填写输入、输出路径、前置依赖和验收条件。能把这四项说清楚，才适合进一步自动建图。


## 来源与延伸阅读

主要来源为 [本地课程课件压缩包](D:/baidudisk/BaiduNetdiskDownload/1、课件(1)/1.课件.rar) 中的《05-实战篇.md》，覆盖环境搭建、工作流设计、日常运行、持续进化、监控维护五个阶段及原文 25 项能力。

持久产物目录、批次校验、稿件摘要审批、PR 交付和健康检查改写属于本篇为解释完整流程补充的设计示例。它们不是 Hermes 已为本机安装完成的系统。

| 内容 | 官方资料 |
| --- | --- |
| Profile 与消息入口 | [Profiles](https://hermes-agent.nousresearch.com/docs/user-guide/profiles/)、[Messaging](https://hermes-agent.nousresearch.com/docs/user-guide/messaging/) |
| Kanban、拆解与 Swarm | [Kanban](https://hermes-agent.nousresearch.com/docs/user-guide/features/kanban/) |
| 子任务委派 | [Delegation](https://hermes-agent.nousresearch.com/docs/user-guide/features/delegation/) |
| 定时任务与 context_from | [Cron](https://hermes-agent.nousresearch.com/docs/user-guide/features/cron/) |
| Skills 与维护 | [Skills](https://hermes-agent.nousresearch.com/docs/user-guide/features/skills/)、[Curator](https://hermes-agent.nousresearch.com/docs/user-guide/features/curator/) |
| MCP | [MCP](https://hermes-agent.nousresearch.com/docs/user-guide/features/mcp/) |
| Hook 协议与计量事件 | [Event Hooks](https://hermes-agent.nousresearch.com/docs/user-guide/features/hooks/) |
| 插件注册 | [开发者插件指南](https://hermes-agent.nousresearch.com/docs/developer-guide/plugins/) |
| Dashboard | [Dashboard](https://hermes-agent.nousresearch.com/docs/user-guide/features/dashboard/) |
| HTTP 请求入口 | [API Server](https://hermes-agent.nousresearch.com/docs/user-guide/features/api-server/) |

主要接口核对使用公开 main 的 [Kanban Schema](https://github.com/NousResearch/hermes-agent/blob/main/tools/kanban_tools_schemas.py)、[Kanban CLI 解析器](https://github.com/NousResearch/hermes-agent/blob/main/hermes_cli/kanban_parser.py)、[委派工具](https://github.com/NousResearch/hermes-agent/blob/main/tools/delegate_tool.py)、[Cron 工具](https://github.com/NousResearch/hermes-agent/blob/main/tools/cronjob_tools.py)、[Skill 管理工具](https://github.com/NousResearch/hermes-agent/blob/main/tools/skill_manager_tool.py)、[Shell Hook](https://github.com/NousResearch/hermes-agent/blob/main/agent/shell_hooks.py) 与 [Profile 解析器](https://github.com/NousResearch/hermes-agent/blob/main/hermes_cli/subcommands/profile.py)。读取日期：2026-10-09。

全部命令与代码均在 Markdown 围栏中。示例经过笔记格式、结构与静态语法检查，没有执行课程功能。
